In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import numpy as np 
from scipy import optimize
from scipy.optimize import fsolve
import numpy as np
from vmbpy import *
from matplotlib import pyplot as plt



In [ ]:
    
def gaussian(x, amplitude, mean, waist, background):
    """1D Gaussian function with a background offset."""
    return amplitude * np.exp(-2 * ((x - mean) / waist)**2) + background

def beam_equations(x, waist1, waist2, distance, lambda_square_in_meter_div_pi_square):
    f1 = x[0]*np.sqrt(1+lambda_square_in_meter_div_pi_square/(x[0]**4)*(x[1]**2)) - waist1
    f2 = x[0]*np.sqrt(1+lambda_square_in_meter_div_pi_square/(x[0]**4)*((x[1]+distance)**2)) - waist2
    return [f1, f2]

def get_frames(n):
    with VmbSystem.get_instance() as vmb:
        cams = vmb.get_all_cameras()
        if not cams:
            print("No cameras found.")
            return None

        with cams[0] as cam:
            print("Using camera:", cam.get_name())
            captured_frames = []

            for i in range(n):
                frame = cam.get_frame()

                # FIX 1: Retain high dynamic range using Mono16
                # This prevents the artificial flat-topping from 8-bit compression
                frame_mono16 = frame.convert_pixel_format(PixelFormat.Mono16)
                img = frame_mono16.as_opencv_image()
                
                captured_frames.append(img)
            
            return np.array(captured_frames)

In [3]:
# target waist in microns

w_input = 5.1e-6
f = 0.02
wavelength = 1550e-9

w_output = wavelength * f / (np.pi * w_input)
print("Calculated output waist:", w_output)



Calculated output waist: 0.0019348247983720609


In [ ]:
# 1. DEFINE YOUR CAMERA'S PIXEL PITCH IN MICROMETERS
# You must look up your specific camera sensor's spec sheet to get this number.
PIXEL_PITCH_UM = 15

def extract_waist_from_frames(frames, pixel_pitch):
    avg_frame = np.mean(frames, axis=0).squeeze()
    h, w = avg_frame.shape
    
    # FIX 2: Find the true center of mass, immune to flat-top clipping
    # Subtracting the median background stops ambient noise from dragging the center
    background = np.median(avg_frame)
    clean_frame = np.maximum(avg_frame - background, 0)
    
    y_center_float, x_center_float = center_of_mass(clean_frame)
    y_center = int(round(y_center_float))
    x_center = int(round(x_center_float))
    
    profile_x = avg_frame[y_center, :]  
    profile_y = avg_frame[:, x_center]  
    
    x_coords = np.arange(w) * pixel_pitch
    y_coords = np.arange(h) * pixel_pitch
    
    p0_x = [np.max(profile_x), x_coords[x_center], (w * pixel_pitch) * 0.1, np.min(profile_x)]
    p0_y = [np.max(profile_y), y_coords[y_center], (h * pixel_pitch) * 0.1, np.min(profile_y)]
    
    fit_x, _ = optimize.curve_fit(gaussian, x_coords, profile_x, p0=p0_x)
    fit_y, _ = optimize.curve_fit(gaussian, y_coords, profile_y, p0=p0_y)
    
    return fit_x[2], fit_y[2]



In [8]:
# get frames at near location 1
frames_location_1 = get_frames(4)

Using camera: Allied Vision Goldeye G-034 (4068040) (DEV_000F31F42DEB)


In [17]:
# get frames at near location 1
frames_location_1 = get_frames(4)
waist_near_x_um, waist_near_y_um = extract_waist_from_frames(frames_location_1, PIXEL_PITCH_UM)
# 3. Convert micrometers to meters for the propagation math
waist_near_x = waist_near_x_um * 1e-6
waist_near_y = waist_near_y_um * 1e-6
print('Waist near X (meters):', waist_near_x)
print('Waist near Y (meters):', waist_near_y)

Using camera: Allied Vision Goldeye G-034 (4068040) (DEV_000F31F42DEB)


RuntimeError: Optimal parameters not found: Number of calls to function has reached maxfev = 1000.

In [11]:
# get frames at far location 2
frames_location_2 = get_frames(4)

Using camera: Allied Vision Goldeye G-034 (4068040) (DEV_000F31F42DEB)


In [10]:

# 2. Extract the near and far waists in micrometers
waist_near_x_um, waist_near_y_um = extract_waist_from_frames(frames_location_1, PIXEL_PITCH_UM)
waist_far_x_um, waist_far_y_um = extract_waist_from_frames(frames_location_2, PIXEL_PITCH_UM)

# 3. Convert micrometers to meters for the propagation math
waist_near_x = waist_near_x_um * 1e-6
waist_near_y = waist_near_y_um * 1e-6
waist_far_x = waist_far_x_um * 1e-6
waist_far_y = waist_far_y_um * 1e-6

print('Waist near X (meters):', waist_near_x)
print('Waist near Y (meters):', waist_near_y)
print('Waist far X (meters):', waist_far_x)
print('Waist far Y (meters):', waist_far_y)

# ==========================================
# YOUR EXISTING FSOLVE CALCULATION
# ==========================================

lambda_square_in_meter_div_pi_square = (1550e-9)**2 / (np.pi)**2
distance = 0.102  # Distance between the two measurement points in meters
g_waist = 1e-5
g_location = 1e-2

calculated_waist_x = fsolve(beam_equations, [g_waist, g_location], args=(waist_near_x, waist_far_x, distance, lambda_square_in_meter_div_pi_square))
calculated_waist_y = fsolve(beam_equations, [g_waist, g_location], args=(waist_near_y, waist_far_y, distance, lambda_square_in_meter_div_pi_square))

# calculated_waist[0] is the waist (w0), calculated_waist[1] is the distance (z)
print('Calculated waist X (meters):', calculated_waist_x[0])
print('Calculated waist Y (meters):', calculated_waist_y[0])

Waist near X (meters): 0.00115595898617834
Waist near Y (meters): 0.0018414699048305645
Waist far X (meters): 0.0016665844195886106
Waist far Y (meters): 0.001672102673229447
Calculated waist X (meters): 9.8307646040761e-05
Calculated waist Y (meters): 0.00029296346987769955


In [ ]:
#Calculated output waist target: 0.0019348247983720609
